# LangChainとは

LangChainとは、大規模言語モデル（LLM）を活用したアプリケーション開発を支援するオープンソースフレームワークのことである。

LangChainについて学習したこと、LangChainを使ったRAGのサンプルコードを記載する。

## 前提条件
- 動作確認は[Google Colab](https://colab.research.google.com)で実施している
  - Google Colab内のパッケージが更新されると動作しない可能性あり
- サンプルコードはGemini APIの無料枠を利用している
  - **無料枠はモデルの学習に利用される**
- [LangChainとLangGraphによるRAG・AIエージェント［実践］入門](https://www.amazon.co.jp/LangChain%E3%81%A8LangGraph%E3%81%AB%E3%82%88%E3%82%8BRAG%E3%83%BBAI%E3%82%A8%E3%83%BC%E3%82%B8%E3%82%A7%E3%83%B3%E3%83%88%EF%BC%BB%E5%AE%9F%E8%B7%B5%EF%BC%BD%E5%85%A5%E9%96%80-%E3%82%A8%E3%83%B3%E3%82%B8%E3%83%8B%E3%82%A2%E9%81%B8%E6%9B%B8-%E8%A5%BF%E8%A6%8B-%E5%85%AC%E5%AE%8F/dp/4297145308)をベースに学習した
  - そのため、古い知識が混ざっている可能性がある

## 注意点
LangChainは以下などの理由からあまり推奨されていないらしい..
- 高度に抽象化されており、知識の応用が効かない(LangChain独特の知識)
- 学習コストが高い
- 性能懸念がある
- RAGの精度を上げたい場合は結局SDKを使った方が良い

## LangChainの機能

LangChainの機能について記載する。  
他にも主だった機能は存在するが、サンプルコードに関連する一部を記載。

| 項目 | 概要 |
| --- | --- |
| プロンプト管理 | プロンプトの管理や最適化を行う機能 |
| Chains機能 | 複数のプロンプトやLLMの処理、出力解析を連結して実行できる機能 |
| LangChain Expression Language(LCEL) | Chains機能を簡単に実現するための記法 |

### プロンプト管理

プロンプトの管理や最適化を行う機能。  
例えば、以下のようにプロンプトをテンプレート化して利用することができる。

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """以下の料理のレシピを考えてください
    料理名: {dish}"""
)

prompt_value = prompt.invoke({"dish": "カレー"})
print(prompt_value)

text='以下の料理のレシピを考えてください\n    料理名: カレー'


---

### Chains機能

複数のプロンプトやLLMの処理、出力解析を連結して実行できる機能。

プロンプトの出力結果を次のプロンプトの入力として渡すことが可能である。  

Chainsには、以下の機能がある。  

| 機能 | 概要 |
| --- | --- |
| Simple Chain | 入力に対して単一のプロンプトとLLMを適用する最も基本的な構成 |
| Sequential Chain | 複数のChainを連結したもの |
| Custom Chain | 処理の流れや分岐を自由に連結したもの |

&nbsp;
それぞれのイメージは以下の通りである。(Gemini作成)  
<img src="./images/LangChain_Chains.png" width="70%">

### LangChain Expression Language(LCEL)

Chains機能を簡単に実現するための記法。

LCELを使わない場合以下のように記述する。

```
# プロンプト生成
prompt_value = prompt.invoke({"dish": "カレー"})
# LLMの実行
ai_message = model.invoke(prompt_value)
# 出力の準備と整形
output_parser = StrOutputParser()
output = output_parser.invoke(ai_message)
```

LCELを使うと以下のように | を使って記述できる。  

```
chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "カレー"})
```

利用例を以下に示す。

In [ ]:
# 不要なパッケージのアンインストール（他のパッケージと依存関係がぶつかるため）
!pip uninstall -y google-adk opentelemetry-exporter-otlp-proto-http
# 必要なパッケージのインストール
!pip install -q -U langchain-google-genai

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# プロンプト生成
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "ユーザが入力した料理のレシピを考えてください。"),
        ("human", "{dish}")
    ]
)

# LLMの準備
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にGoogle Colabのシークレットに登録しておく
    )

chain = prompt | model | StrOutputParser()
output = chain.invoke({"dish": "カレー"})
print(output)

ご家庭で手軽に作れる、定番の美味しいカレーのレシピをご紹介します。市販のカレールーを使って、誰でも失敗なく作れるように工夫しました。

---

## 定番！家庭で作る美味しいポークカレー

じっくり炒めた玉ねぎの甘みと、お肉の旨味が溶け込んだ、ご飯が進む一品です。市販のカレールーを使うので、カレー作り初心者さんでも安心！

### 材料（4人分）

*   **豚肉（カレー用またはこま切れ）**: 300g
    *   ※豚バラブロック肉を使うと、よりコクが出ます。
*   **玉ねぎ**: 2個
*   **じゃがいも**: 2個
*   **にんじん**: 1本
*   **市販のカレールー**: 1箱（8皿分など、商品の表示に従う）
*   **水**: ルーの表示に従う（例：800ml）
*   **サラダ油**: 大さじ1
*   **隠し味（お好みで）**:
    *   すりおろしりんご: 大さじ2
    *   はちみつ: 大さじ1
    *   インスタントコーヒー: 小さじ1/2
    *   ウスターソース: 大さじ1

### 作り方

1.  **下準備をする**
    *   豚肉は食べやすい大きさに切る。
    *   玉ねぎは薄切り、またはくし切りにする。
    *   じゃがいも、にんじんは皮をむき、乱切りにする。じゃがいもは水にさらしてアクを抜いておく。
    *   カレールーは細かく刻んでおくか、手で割っておくと溶けやすい。

2.  **具材を炒める**
    *   厚手の鍋にサラダ油を熱し、玉ねぎを入れ、しんなりするまで中火で炒める。
        *   **ポイント**: 時間があれば、玉ねぎを飴色になるまでじっくり炒めると、甘みとコクが格段にアップします。
    *   玉ねぎが透き通ってきたら、豚肉を加えて色が変わるまで炒める。
    *   にんじん、じゃがいもを加えて、全体に油が回るまで炒め合わせる。

3.  **煮込む**
    *   水を加え、沸騰したら丁寧に出てくるアクを取り除く。
    *   蓋をして、弱火でじゃがいもとにんじんが柔らかくなるまで20分～30分煮込む。

4.  **ルーを加える**
    *   一度火を止め、カレールーを少しずつ加えながら、

---

## RAGのお試し

LangChainでは、RAGに使用するための以下の主要コンポーネントが存在する。

| コンポーネント | 概要
| --- | --- |
| Document loader| 様々なデータソースからドキュメントを読み込む |
| Document transformer | ドキュメントに何らかの変換をかける |
| Embedding model | ドキュメントをベクトル化する |
| Vector store | ベクトル化したドキュメントの保存先 |
| Retriever | 入力のテキストと関連するドキュメントを検索する |

&nbsp;   
それぞれのコンポーネントのつながりは以下のイメージである。(Gemini作成)  

<img src="./images/LangChain_RAG_Component.png" width="70%">

### Document loader

様々なデータソースからドキュメントを読み込むためのコンポーネント。

| Document loader | 概要 |
| --- | --- |
| GitLoader | リポジトリからファイルを読み込む |
| ConfluenceLoader | Confluenceのページを読み込み |
| UnstructuredLoader | テキストファイル、パワーポイント、HTML、PDF、画像などのファイルを読み込む |
| DirectoryLoader | ディレクトリ内のファイルをUnstructuredLoaderなどで読み込む |
| S3DirectoryLoader | Amazon S3のバケットを指定してオブジェクトを読み込む |

GitLoaderの例を以下に示す。  
※ 試しに「https://github.com/junit-team/junit-framework/tree/main/junit-jupiter-params」 のJavaファイルを対象に取得

In [ ]:
# 必要なパッケージのインストール
# 実行後セッションの再起動が必要
!pip install -q -U --force-reinstall langchain langchain-community langchain-chroma langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str) -> bool:
  return "junit-jupiter-params" in file_path and file_path.endswith(".java")

loader = GitLoader(
    clone_url="https://github.com/junit-team/junit-framework",
    repo_path="./junit-framework",
    branch="main",
    file_filter=file_filter,
)

raw_docs = loader.load()

print(f"ファイル名: {raw_docs[0].metadata.get("file_path", "不明")}")
print(f"先頭50文字: {raw_docs[0].page_content[:50]}...")

ファイル名: junit-jupiter-params/src/main/java/module-info.java
先頭50文字: /*
 * Copyright 2015-2026 the original author or a...


---

### Document transformer

Document loaderで読み込んだドキュメントに変換をかけるためのコンポーネント。  
例えば、ドキュメントをある程度のチャンクに分割できる。

利用例を以下に示す。

In [ ]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

java_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.JAVA,
    chunk_size=1000,  # チャンクの最大サイズ
    chunk_overlap=10  # チャンク間の重複サイズ
)

# コードの分割実行
docs = java_splitter.split_documents(raw_docs)
print(docs[:2])

[Document(metadata={'source': 'junit-jupiter-params/src/main/java/module-info.java', 'file_path': 'junit-jupiter-params/src/main/java/module-info.java', 'file_name': 'module-info.java', 'file_type': '.java'}, page_content='/*\n * Copyright 2015-2026 the original author or authors.\n *\n * All rights reserved. This program and the accompanying materials are\n * made available under the terms of the Eclipse Public License v2.0 which\n * accompanies this distribution and is available at\n *\n * https://www.eclipse.org/legal/epl-v20.html\n */\n\n/**\n * JUnit Jupiter extension for parameterized tests.\n *\n * @since 5.0\n */\nmodule org.junit.jupiter.params {\n\n\trequires static transitive org.apiguardian.api;\n\trequires static transitive org.jspecify;\n\n\trequires transitive org.junit.jupiter.api;\n\trequires transitive org.junit.platform.commons;\n\n\texports org.junit.jupiter.params;\n\texports org.junit.jupiter.params.aggregator;\n\texports org.junit.jupiter.params.converter;\n\texp

---

### Embedding model

ドキュメントの変換処理後に、ドキュメントをベクトル化するためのコンポーネント。

利用例を以下に示す。

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Gemini Embeddingモデルの設定
# 無料枠で利用可能な 'models/gemini-embedding-001' を使用
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",  # ドキュメント登録用に最適化
    google_api_key=userdata.get("GEMINI_API_KEY") # 事前にGoogle Colabのシークレットに登録しておく
)

# ベクトル化の例（多次元に変換されるため10個まで表示）
print(embeddings.embed_query("任意の文字列をベクトル化する")[:10])

[-0.01880163, -0.004134218, 0.010412937, -0.08069568, 0.012144683, 0.023088612, 0.011634183, 0.012314894, 0.033910364, 0.013011293]


---

### Vector store/Retriever

Vector Storeはベクトル化したドキュメントの保存先のこと。  
Retrieverは入力のテキストと関連するドキュメントを検索するのこと。

Vector StoreにLangChainと親和性のあるOSSであるChromaを利用した例を以下に示す。

In [ ]:
from langchain_chroma import Chroma

# Vector storeに保存（複数ドキュメントを対象にリクエストすると無料枠の上限にひっかかるため対象を減らす）
db = Chroma.from_documents(docs[:50], embeddings)

# Retrieverを使って関連するドキュメントが検索できることを確認
# パラメータ化テストは複数の入力データと期待される結果セットを渡して繰り返しテストが行える機能
retriever = db.as_retriever()
context_docs = retriever.invoke("パラメータ化テストに利用するアノテーションを教えてください")
print(context_docs[0].page_content)

public @interface AfterParameterizedClassInvocation {

	/**
	 * Whether the arguments of the parameterized test class should be injected
	 * into the annotated method (defaults to {@code true}).
	 */
	boolean injectArguments() default true;

}


---

LLMを使ってVector Storeに問い合わせる。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# プロンプト生成
prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください

文脈： """
{context}
"""

質問: {question}
''')

# LLMの準備
model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        google_api_key=userdata.get("GEMINI_API_KEY") # 事前にGoogle Colabのシークレットに登録しておく
    )

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

output = chain.invoke("パラメータ化テストに利用するアノテーションを教えてください")
print(output)

提供された文脈からは、以下の2つのアノテーションがパラメータ化テストに関連して利用されることがわかります。

*   `@AfterParameterizedClassInvocation`
*   `@BeforeParameterizedClassInvocation`
